# Train an IGNODE-compatible detector with RF-DETR (Small)

**Audience:** customers who want a transformer-based alternative to YOLOX. RF-DETR can outperform YOLO on cluttered scenes with many small overlapping objects, at the cost of slower training and inference.

**Output:** an ONNX model + sidecar JSON files ready for **Custom Model Upload** in your IGNODE workspace.

## When to choose RF-DETR over YOLOX
- ✅ Dense scenes (>20 objects per image)
- ✅ Small overlapping objects (cells under a microscope, tiny defects)
- ✅ You have ≥500 training images per class
- ❌ Limited training data (<100 images per class) → use YOLOX
- ❌ Real-time on edge devices → use YOLOX-nano or YOLOX-tiny

## Architecture decoder hint for IGNODE
RF-DETR exports use the `detr` decoder family in IGNODE UINF (`agent_runner/frameworks/detection_decoders/detr.py`). The sidecar declares `postprocess.family: 'detr'` — UINF parses logits + boxes from the DETR output shape and applies post-NMS filtering.

In [ ]:
# Step 0 — install RF-DETR + ONNX export deps. ~3 min on cold runtime.
!pip install -q rfdetr supervision==0.21.0 onnx==1.21.0 onnxruntime==1.23.2

In [ ]:
# Step 1 — point at your dataset (COCO JSON layout).
# RF-DETR expects: <root>/train/_annotations.coco.json + <root>/train/*.jpg
#                  <root>/valid/_annotations.coco.json + <root>/valid/*.jpg
# If you have VOC or YOLO, convert via supervision first (see notes below).
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/my-detection-dataset'
WORKDIR     = '/content/rfdetr-run'
!mkdir -p {WORKDIR}
!ls -la {DATASET_DIR}

In [ ]:
# Step 2 — train RF-DETR Small. The Small variant balances accuracy and
# speed; Base and Large variants exist with --model rfdetr_base /
# rfdetr_large for better mAP at higher VRAM cost.
from rfdetr import RFDETRSmall

model = RFDETRSmall()  # downloads COCO-pretrained weights on first use
model.train(
    dataset_dir=DATASET_DIR,
    epochs=100,                  # RF-DETR converges faster than YOLOX; 100 is usually enough
    batch_size=8,                # RF-DETR is memory-hungry; lower than YOLOX's 16
    grad_accum_steps=4,          # effective batch = 32
    lr=1e-4,                     # transformer learning rate; lower than YOLO's 5e-3
    output_dir=WORKDIR,
    early_stopping=True,         # safer with transformer plateaus
    early_stopping_patience=10,
)

In [ ]:
# Step 3 — export to ONNX. RF-DETR's exporter handles the unfix-shape
# inputs (dynamic batch + image size) and emits a model UINF's `detr`
# decoder accepts: outputs are (logits, boxes) tuple at fixed indices.
import pathlib
ONNX_OUT = pathlib.Path(WORKDIR) / 'model.onnx'
model.export(
    output=str(ONNX_OUT),
    opset=18,
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}, 'boxes': {0: 'batch'}},
)
print('Exported:', ONNX_OUT)

In [ ]:
# IR-3.S.A — Step 4: write sidecars + auto-derive CLASS_LABELS.
# DO NOT hand-type class names here. RF-DETR's class index 0 is
# the FIRST category in your COCO JSON (sorted by category id);
# typing them in the wrong order ships a model where every
# prediction's label is rotated. The cell below derives the list
# from the dataset itself, then the next cell smoke-tests the
# exported ONNX so you SEE the index→label map before uploading.
import json, shutil, pathlib

# 1. Pull class names directly from the training COCO JSON. Sorted
# by category id so the ordering matches what RF-DETR learned.
_train_json = pathlib.Path(DATASET_DIR) / 'train' / '_annotations.coco.json'
_coco = json.loads(_train_json.read_text())
_cats = sorted(_coco['categories'], key=lambda c: c.get('id', 0))
CLASS_LABELS = [str(c['name']) for c in _cats]
print(f'Derived CLASS_LABELS from {_train_json}:')
for i, name in enumerate(CLASS_LABELS):
    print(f'  [{i}] {name}')
print()
print('If this order surprises you, STOP and inspect the JSON before bundling.')
print('Mislabeled here = wrong predictions at deploy time.')

preprocess_config = {
    'input_size':      [800, 800],
    'mean':            [0.485, 0.456, 0.406],
    'std':             [0.229, 0.224, 0.225],
    'channel_order':   'RGB',
    'image_format':    'CHW',
    'rescale':         'imagenet',
    'resize_method':   'letterbox',
    'letterbox_color': [114, 114, 114],
    'postprocess': {
        'family':               'detr',
        'nms_required':         False,            # DETR is set-prediction; no NMS by design
        'confidence_threshold': 0.5,
        'num_queries':          300,              # RF-DETR Small's default query slot count
    },
    '_backbone': 'rfdetr_small',
}

out = pathlib.Path(WORKDIR) / 'upload-bundle'
out.mkdir(exist_ok=True)
shutil.copy(ONNX_OUT, out / 'model.onnx')
(out / 'preprocess_config.json').write_text(json.dumps(preprocess_config, indent=2))
(out / 'class_labels.json').write_text(json.dumps(CLASS_LABELS, indent=2))

print()
print('Upload bundle ready at:', out)
!ls -la {out}


In [ ]:
# IR-3.S.A — Step 4.5: ONNX smoke-test (DO NOT SKIP).
# Runs the freshly-exported ONNX on ONE validation image and prints
# the highest-confidence query's (class_index, label) pair. If the
# top prediction is labeled in a way that obviously contradicts the
# image's content, your class_labels.json is in the wrong order —
# fix it BEFORE uploading.
import json, pathlib, numpy as np, onnxruntime as ort
from PIL import Image

_bundle = pathlib.Path(WORKDIR) / 'upload-bundle'
_labels = json.loads((_bundle / 'class_labels.json').read_text())
_sess   = ort.InferenceSession(str(_bundle / 'model.onnx'), providers=['CPUExecutionProvider'])
_inp    = _sess.get_inputs()[0].name

# Pick the first validation image. Change this path if your layout differs.
_val_dir   = pathlib.Path(DATASET_DIR) / 'valid'
_test_imgs = sorted([p for p in _val_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
assert _test_imgs, f'No images in {_val_dir} — adjust DATASET_DIR or the val split name.'
_img_path = _test_imgs[0]
print(f'Smoke-testing on: {_img_path.name}')

# Mirror Step 4 preprocess so the input matches what UINF will feed.
_pil = Image.open(_img_path).convert('RGB').resize((800, 800), Image.BILINEAR)
_arr = (np.asarray(_pil, dtype=np.float32) / 255.0 - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
_arr = _arr.transpose(2, 0, 1)[None, ...]

_outs   = _sess.run(None, {_inp: _arr})
_logits = next((o for o in _outs if o.ndim == 3 and o.shape[-1] >= len(_labels)), None)
if _logits is None:
    print('WARN: could not find logits output — skipping smoke-test.')
else:
    # RF-DETR logits include a no-object slot at index len(classes); take the top class slot only.
    _probs = 1.0 / (1.0 + np.exp(-_logits[0, :, :len(_labels)]))
    _best_query = int(_probs.max(axis=1).argmax())
    _best_class = int(_probs[_best_query].argmax())
    _best_score = float(_probs[_best_query, _best_class])
    print()
    print(f'Top-confidence query: idx={_best_class} → class_labels[{_best_class}] = {_labels[_best_class]!r}')
    print(f'Confidence: {_best_score:.3f}')
    print()
    print('Sanity check: open the image above. If the model thinks it is')
    print(f'a {_labels[_best_class]!r} but you can see it is something else, STOP and fix')
    print('class_labels.json before continuing. Look in upload-bundle/class_labels.json.')


## Step 5 — upload to IGNODE

Same as YOLOX:
1. ML Factory → Models → **Upload custom model**
2. Drag the 3 files from `upload-bundle/` into the modal
3. Deploy to a UINF instance
4. Verify in Playground

## Troubleshooting
- **All predictions get filtered at threshold 0.5**: DETR's confidence is calibrated differently than YOLO. Try 0.3 or 0.2 in the Playground first; tighten once you see real predictions.
- **Wrong class names**: Check `class_labels.json` matches your dataset's COCO `categories[].name` order EXACTLY. UINF maps output index → list index.
- **Boxes outside image**: RF-DETR outputs normalized coords (0..1); UINF's `detr` decoder de-normalizes against `image_size`. If your input dims differ from 800×800, override `input_size` in `preprocess_config.json`.